# NOTEBOOK 05 — FEATURE ENGINEERING
## Country-level pipeline

### Mục tiêu
- Chuyển insight EDA thành feature cho country.
- Tạo các biến phái sinh từ target và các bảng phụ.
- Không dùng latitude/longitude.
- Lưu `climate_country_features`.

### Mốc đánh giá
- Train/reference: đến hết **2003**
- Test: **2004 trở đi**


## I. Setup và đọc dữ liệu sạch

In [ ]:
from pathlib import Path
import os

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS = PROJECT_ROOT / "artifacts"
APP_DIR = PROJECT_ROOT / "app"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)
APP_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url_object = URL.create(
    "postgresql",
    username="postgres",
    password="123456",
    host="localhost",
    port=5432,
    database="climate_change_db1",
)
engine = create_engine(url_object, pool_pre_ping=True)

with engine.connect() as conn:
    print("Database:", conn.execute(text("SELECT current_database()")).scalar())

In [ ]:
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
from IPython.display import display

df = pd.read_sql(
    text("SELECT * FROM climate_country_clean ORDER BY country, dt"),
    engine,
    parse_dates=["dt"]
)

REFERENCE_END = pd.Timestamp("2003-12-01")
reference = df[df["dt"] <= REFERENCE_END].copy()

df["year"] = df["dt"].dt.year
df["month"] = df["dt"].dt.month
df["quarter"] = df["dt"].dt.quarter

print("Full range:", df["dt"].min(), "→", df["dt"].max())
print("Reference ends:", REFERENCE_END.date())

## II. Time features

In [ ]:
feat = df.copy()

feat["time_idx"] = (
    (feat["year"] - 1850) * 12
    + (feat["month"] - 1)
)

feat["month_sin"] = np.sin(
    2 * np.pi * feat["month"] / 12
)
feat["month_cos"] = np.cos(
    2 * np.pi * feat["month"] / 12
)


## III. Target history & Rolling features (Shift 1)

In [ ]:
COUNTRY_TEMP = "average_temperature_filled"

# History features
feat["temp_lag_1"] = feat.groupby("country")[COUNTRY_TEMP].shift(1)
feat["temp_lag_3"] = feat.groupby("country")[COUNTRY_TEMP].shift(3)
feat["temp_lag_6"] = feat.groupby("country")[COUNTRY_TEMP].shift(6)
feat["temp_lag_12"] = feat.groupby("country")[COUNTRY_TEMP].shift(12)
feat["temp_lag_24"] = feat.groupby("country")[COUNTRY_TEMP].shift(24)

# Rolling features (must use shift(1) to prevent leakage)
feat["temp_roll_mean_3"] = feat.groupby("country")[COUNTRY_TEMP].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
feat["temp_roll_mean_6"] = feat.groupby("country")[COUNTRY_TEMP].transform(lambda s: s.shift(1).rolling(6, min_periods=1).mean())
feat["temp_roll_mean_12"] = feat.groupby("country")[COUNTRY_TEMP].transform(lambda s: s.shift(1).rolling(12, min_periods=1).mean())
feat["temp_roll_std_12"] = feat.groupby("country")[COUNTRY_TEMP].transform(lambda s: s.shift(1).rolling(12, min_periods=2).std())


## IV. Derived Auxiliary Features

In [ ]:
# We create lagged auxiliary features
feat["global_temp_lag_1"] = feat.groupby("country")["global_temperature"].shift(1)
feat["global_temp_lag_12"] = feat.groupby("country")["global_temperature"].shift(12)
feat["global_temp_roll12_lag1"] = feat.groupby("country")["global_temperature"].transform(lambda s: s.shift(1).rolling(12, min_periods=1).mean())

feat["city_summary_temp_lag_1"] = feat.groupby("country")["city_country_avg_temperature"].shift(1)
feat["city_summary_temp_lag_12"] = feat.groupby("country")["city_country_avg_temperature"].shift(12)
feat["city_summary_temp_roll12_lag1"] = feat.groupby("country")["city_country_avg_temperature"].transform(lambda s: s.shift(1).rolling(12, min_periods=1).mean())

feat["state_summary_temp_lag_1"] = feat.groupby("country")["state_country_avg_temperature"].shift(1)
feat["state_summary_temp_lag_12"] = feat.groupby("country")["state_country_avg_temperature"].shift(12)
feat["state_summary_temp_roll12_lag1"] = feat.groupby("country")["state_country_avg_temperature"].transform(lambda s: s.shift(1).rolling(12, min_periods=1).mean())

feat["major_city_temp_lag_1"] = feat.groupby("country")["major_city_avg_temperature"].shift(1)
feat["major_city_temp_lag_12"] = feat.groupby("country")["major_city_avg_temperature"].shift(12)
feat["major_city_temp_roll12_lag1"] = feat.groupby("country")["major_city_avg_temperature"].transform(lambda s: s.shift(1).rolling(12, min_periods=1).mean())

# Gap features
feat["city_country_gap_lag1"] = (feat["city_country_avg_temperature"] - feat[COUNTRY_TEMP]).groupby(feat["country"]).shift(1)
feat["state_country_gap_lag1"] = (feat["state_country_avg_temperature"] - feat[COUNTRY_TEMP]).groupby(feat["country"]).shift(1)
feat["major_city_country_gap_lag1"] = (feat["major_city_avg_temperature"] - feat[COUNTRY_TEMP]).groupby(feat["country"]).shift(1)
feat["country_global_gap_lag1"] = (feat[COUNTRY_TEMP] - feat["global_temperature"]).groupby(feat["country"]).shift(1)


## V. Kiểm tra số derived features từ bảng phụ

In [ ]:
AUX_FEATURES = [
    "global_temp_lag_1", "global_temp_lag_12", "global_temp_roll12_lag1",
    "city_summary_temp_lag_1", "city_summary_temp_lag_12", "city_summary_temp_roll12_lag1",
    "state_summary_temp_lag_1", "state_summary_temp_lag_12", "state_summary_temp_roll12_lag1",
    "major_city_temp_lag_1", "major_city_temp_lag_12", "major_city_temp_roll12_lag1",
    "city_country_gap_lag1", "state_country_gap_lag1", "major_city_country_gap_lag1", "country_global_gap_lag1"
]

print("Derived auxiliary features:", len(AUX_FEATURES))
assert len(AUX_FEATURES) >= 10


## VI. Feature set dùng cho Modeling\n\nTất cả feature sau đều future-safe / past-data only.

In [ ]:
TIME_FEATURES = [
    "year", "month", "quarter", "time_idx", "month_sin", "month_cos"
]

TARGET_HISTORY_FEATURES = [
    "temp_lag_1", "temp_lag_3", "temp_lag_6", "temp_lag_12", "temp_lag_24",
    "temp_roll_mean_3", "temp_roll_mean_6", "temp_roll_mean_12", "temp_roll_std_12"
]

FEATURES = TIME_FEATURES + TARGET_HISTORY_FEATURES + AUX_FEATURES

FORECAST_SAFE_FEATURES = FEATURES.copy()

print("Total numeric features:", len(FEATURES))
print(FEATURES)


## VII. Quality audit của features

In [ ]:
quality = pd.DataFrame({
    "dtype": feat[FEATURES].dtypes.astype(str),
    "missing_count": feat[FEATURES].isna().sum(),
    "missing_pct": (feat[FEATURES].isna().mean() * 100).round(2),
    "nunique": feat[FEATURES].nunique(dropna=False)
}).sort_values("missing_pct", ascending=False)

display(quality)


## VIII. Feature selection / registry\n\nNote: Country is the categorical modeling entity, and will be encoded in notebook 06.

In [ ]:
TARGET_COUNTRIES = sorted(feat["country"].dropna().unique())

feature_registry = pd.DataFrame({
    "feature": FEATURES,
    "source": (
        ["time"] * len(TIME_FEATURES)
        + ["target_history"] * len(TARGET_HISTORY_FEATURES)
        + ["auxiliary"] * len(AUX_FEATURES)
    ),
    "future_safe": True
})

display(feature_registry)

registry_payload = {
    "reference_end": str(REFERENCE_END.date()),
    "time_features": TIME_FEATURES,
    "target_history_features": TARGET_HISTORY_FEATURES,
    "aux_features": AUX_FEATURES,
    "features": FEATURES,
    "countries": TARGET_COUNTRIES
}

(ARTIFACTS / "feature_registry.json").write_text(
    json.dumps(
        registry_payload,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


## IX. Lưu `climate_country_features`

In [ ]:
feat.to_sql(
    "climate_country_features",
    engine,
    if_exists="replace",
    index=False,
    chunksize=20_000,
    method="multi"
)

with engine.begin() as conn:
    conn.execute(text(
        "CREATE INDEX IF NOT EXISTS idx_climate_country_features_country_dt "
        "ON climate_country_features(country, dt)"
    ))

print("Saved feature tables.")
